# ZuCo / TeCo Task-1 — Spatial Sentiment Decoding (thin Colab driver)

Pipeline code lives in the **eeg-spatial-encoding** GitHub repo. This notebook clones/updates
it and runs it. Re-run the **Setup** cell to pick up code changes. Run **Setup -> mount Drive
-> Config -> Build**, then any analysis section. The **TeCo** section at the bottom builds the
Persian-dataset features the same way.


## Setup — pull latest code from GitHub + install deps


In [ ]:
import os, sys
REPO_URL = 'https://github.com/parmisbathaeiyan/eeg-spatial-encoding.git'
REPO_DIR = '/content/eeg-spatial-encoding'
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
!pip install -q -r {REPO_DIR}/requirements.txt
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('code ready:', REPO_DIR)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Config — ZuCo Drive paths


In [ ]:
from eeg_spatial.config import Config
cfg = Config(
    sr_data_dir='/content/drive/MyDrive/Parmis/Thesis/Data/zuco_og_raw',
    labels_csv='/content/drive/MyDrive/Parmis/Thesis/Data/zuco_sentiment_labels_task1_fixed.csv',
    montage_npz='/content/drive/MyDrive/Parmis/Thesis/Data/zuco_montage.npz',
    feature_prefix='TRT',
    k_neighbors=4,
)
RESULTS_CSV = '/content/drive/MyDrive/Parmis/Thesis/Results/spatial_accuracy/spatial_accuracy_results.csv'


## Build (ZuCo) dataset + montage + neighbor graph


In [ ]:
from eeg_spatial import dataset, montage
label_map = dataset.load_label_map(cfg.labels_csv)
X_band, y, meta = dataset.build_band_dataset(cfg, label_map)
ch_names, ch_pos, mont = montage.load_montage(cfg.montage_npz)
neighbor_idx = montage.build_neighbor_graph(ch_pos, cfg.k_neighbors)
info = montage.make_info(ch_names, mont)
assert X_band.shape[1] == len(ch_names), 'channel-count mismatch between data and montage'


## (Optional) Export ZuCo features for local runs


In [ ]:
import numpy as np
FEATURES_NPZ = '/content/drive/MyDrive/Parmis/Thesis/Data/zuco_features.npz'
np.savez(FEATURES_NPZ, X_band=X_band, y=y, groups=np.asarray(meta['texts']))
print('saved ->', FEATURES_NPZ, '| X_band', X_band.shape)


## Sentence-level searchlight (ZuCo) + topomap


In [ ]:
from eeg_spatial import searchlight, plotting
# add  groups=meta['texts']  for leakage-free CV; add  scoring='balanced'  for balanced accuracy
results_df = searchlight.run_searchlight(
    X_band, y, neighbor_idx, ch_names, n_folds=cfg.n_folds, random_state=cfg.random_state)
searchlight.report_searchlight(results_df)
os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
results_df.to_csv(RESULTS_CSV)
plotting.plot_accuracy_topomaps(results_df, info, cfg.k_neighbors)


## (Optional) Word-concatenated features (no averaging)

Keeps each word's vector (padded to a fixed word count) instead of averaging. Independent of
the section above; saves to a separate CSV.


In [ ]:
from eeg_spatial import dataset, searchlight, plotting
X_concat, y_concat, max_words = dataset.build_concat_dataset(cfg, label_map, max_words='p95')
results_concat = searchlight.run_searchlight(
    X_concat, y_concat, neighbor_idx, ch_names, n_folds=cfg.n_folds, random_state=cfg.random_state)
searchlight.report_searchlight(results_concat)
results_concat.to_csv(RESULTS_CSV.replace('.csv', '_wordconcat.csv'))
plotting.plot_accuracy_topomaps(results_concat, info, cfg.k_neighbors, title_suffix=' (concatenated words)')


## (Optional) 1D-CNN searchlight on raw EEG

Heavy: rebuilds the dataset keeping raw EEG, then trains one CNN per electrode neighborhood.
**Use a GPU runtime.** Adds a `CNN` column to the sentence-level results.


In [ ]:
import torch, pandas as pd
from eeg_spatial import dataset, cnn, plotting
try:
    results_df
except NameError:
    results_df = pd.read_csv(RESULTS_CSV, index_col=0) if os.path.exists(RESULTS_CSV) else pd.DataFrame(index=ch_names)
cfg.keep_raw_eeg = True
X_band, y, meta = dataset.build_band_dataset(cfg, label_map)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
cnn_acc = cnn.run_cnn_searchlight(
    meta['raw_eeg'], y, neighbor_idx, ch_names, device,
    epochs=15, batch_size=16, n_folds=cfg.n_folds, random_state=cfg.random_state)
results_df['CNN'] = cnn_acc
results_df.to_csv(RESULTS_CSV)
plotting.plot_accuracy_topomaps(results_df, info, cfg.k_neighbors, title_suffix=' (incl. CNN)')


## (Optional) Cluster-based permutation test — significance over space

Leakage-free grouped CV by sentence + sentence-level label permutation + spatial cluster
correction. LDA + parallel. Best run locally (scripts/run_local.py).


In [ ]:
from eeg_spatial import stats
perm = stats.permutation_cluster_searchlight(
    X_band, y, neighbor_idx, groups=meta['texts'],
    n_permutations=200, n_folds=cfg.n_folds, random_state=cfg.random_state,
    scoring='balanced', n_jobs=-1)
stats.plot_significance_topomap(perm, info)


## (TeCo) Build Persian-dataset features + montage

Same pipeline on **TeCo** (Persian; 126 channels; 165 sentences, balanced 55/55/55). Reads the
TeCo HDF5 `.mat` files, builds per-sentence TRT band-power features, and exports
`teco_features.npz` + `teco_montage.npz` to Drive. Download those two and run the searchlight /
permutation locally exactly like ZuCo:

```
python scripts/run_local.py --features teco_features.npz --montage teco_montage.npz --grouped --scoring balanced --permutation --clf svm --n-perm 400 --no-plot
```

Needs `location_xyz_reference.txt` (the 126-electrode coordinates) on Drive.


In [ ]:
from eeg_spatial import teco
import numpy as np

# --- TeCo paths on your Drive (verify these) ---
TECO_TASK1_ROOT  = '/content/drive/MyDrive/thesis_data/TeCo/task1_with_total'        # <subject>/task1.mat
TECO_LABELS_CSV  = '/content/drive/MyDrive/thesis_data/teco_sentiment_labels_task1.csv'
TECO_MONTAGE_TXT = '/content/drive/MyDrive/thesis_data/location_xyz_reference.txt'   # 126-ch coords
TECO_OUT_DIR     = '/content/drive/MyDrive/Parmis/Thesis/Data'

mat_files = teco.find_teco_mat_files(TECO_TASK1_ROOT)
print(len(mat_files), 'TeCo subject files')

label_map = teco.load_teco_label_map(TECO_LABELS_CSV)
X_teco, y_teco, meta_teco = teco.build_teco_dataset(mat_files, label_map)   # (N, 126, 8)

ch_names_t, ch_pos_t, mont_t = teco.load_teco_montage(TECO_MONTAGE_TXT)
np.savez(f'{TECO_OUT_DIR}/teco_montage.npz', labels=ch_names_t, coords=ch_pos_t)

np.savez(f'{TECO_OUT_DIR}/teco_features.npz', X_band=X_teco, y=y_teco, groups=np.asarray(meta_teco['texts']))
print('saved teco_features.npz + teco_montage.npz to', TECO_OUT_DIR)
